# DL-0: geometric measurement × one learned number, and which pieces to use

**CIS 831 Deep Learning project, Kansas State University**

This notebook runs end to end: from the videos to the DL-0 results. Everything already done (clicks, tracking, measurements) is re-used, so on a second run it takes minutes.

**Two questions:**
1. **Which pieces should we average?** A piece pointing toward the camera looks shorter than it is, and nothing can make a piece look *longer*. So in each clip the longest good mask is the best view of the true length, and masks close to it are pieces lying flat to the camera. We compare: every mask, every good mask, good masks at least 50 / 60 / 70 / 80 / 90 % as long as the longest, and only the longest.
2. **DL-0:** does one learned correction factor *k* fix the camera's error? Prediction = *k* × geometric value, with *k* learned on other clips.

| Step | What happens | Saved, so it is done only once |
|---|---|---|
| 1–3 | Settings, lab measurements, list of videos | — |
| 4–5 | Clean-die pictures; **click tool only for videos without saved points** | `results/<video>/clean_die.png`, `clicks.json` |
| 6 | Exact hole centres, pixel → mm scale | `clicks.json` |
| 7–8 | SAM 2.1 tracking (SAM 2 is loaded only if a video still needs it) | `results/<video>/cuts.csv`, `masks/` |
| 9 | Every RED and BLUE mask measured | `results/<video>/pieces.csv` |
| 10–12 | Length ratio of every mask; the piece selections | — |
| 13–18 | DL-0, errors, ablation, comparison with the geometric value | `results/dl0/` |

The per-video results are **shared with notebook 02**: whatever one notebook has done, the other re-uses.

### How to run (VS Code)
- Kernel: **Python (extruder-sam2)**. Run the cells one at a time (Shift+Enter).
- **Step 5** shows the click tool only if a video has no saved points. Click, press **Save & next**, then continue.
- First run on new videos: about 2–3 hours (SAM 2 tracking). After that: minutes.

## Step 1: Imports and settings

In [ ]:
import json, os, random, time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display

from extruder import video as vid, measure, pipeline, track, labels as labmod, explain, experiment, dl0
from extruder.ui import MultiVideoClicker

%matplotlib inline
plt.rcParams["figure.dpi"] = 90
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40); pd.set_option("display.max_rows", 200)
print("PyTorch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

In [ ]:
# ---- folders -----------------------------------------------------------------------------
PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_DIR / "data"
VIDEO_DIR = DATA_DIR / "Video from Trial Run 10.07.26"     # the trial videos (901_1.MP4 ...)
EXCLUDE = ("Blank",)                                         # videos whose name contains these are skipped
ONLY_VIDEOS = None                                           # e.g. ["901_1", "903_1"] for a quick test; None = all
RESULTS_DIR = PROJECT_DIR / "results"                        # one folder per video (shared with notebook 02)
OUT_DIR = RESULTS_DIR / "dl0"                                # this notebook's tables and figures

# ---- lab measurements ------------------------------------------------------------------------
LABELS_FILE = DATA_DIR / "Trial Run Data.xlsx"
LABELS_SHEET = "Oct 7 Final"
LABEL_COLUMNS = dict(clip="Sl. No.", piece="Piece no.",
                     length="Length of the Product (mm)", diameter="Diameter of the Product (mm)")

# ---- clicks ----------------------------------------------------------------------------------
SHOW_CLICKER = False   # True = open the click tool even when every video already has saved points

# ---- SAM 2.1 and tracking (same as notebook 02) ---------------------------------------------------
SAM2_CHECKPOINT = PROJECT_DIR / "checkpoints" / "sam2.1_hiera_large.pt"
SAM2_CONFIG = "configs/sam2.1/sam2.1_hiera_l.yaml"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CROP_SIZE = 360
N_TRACK = None          # None = automatic: about 1.6 knife cycles
MAX_CUTS = None         # None = every cut; a number = only the first N cuts per video (quick test)
FORCE_TRACKING = False
USE_BRIGHTNESS = False

# ---- measuring and dropping bad masks (same as notebook 02) ----------------------------------------
N_JOBS = max(1, (os.cpu_count() or 2) - 2)
FORCE_MEASURE = False
measure.MAX_ROUGHNESS_PCT = 2.0
measure.TUBE_FILL_RANGE = (0.80, 1.20)
measure.MAX_STABILITY_PCT = 5.0

# ---- which pieces to use, and DL-0 ------------------------------------------------------------
LENGTH_THRESHOLDS = (0.5, 0.6, 0.7, 0.8, 0.9, 1.0)   # keep good masks at least this fraction of the reference length
LENGTH_REFERENCES = ("longest", "p95")              # reference = longest good mask, or 95th-percentile length
SPLITS = ("leave one setting out",)                 # add "leave one clip out" for the easier extra test

EXAMPLE_CLIP = None    # clip used for the explanation figure; None = the first one
SEED = 0

OUT_DIR.mkdir(parents=True, exist_ok=True); (OUT_DIR / "figures").mkdir(exist_ok=True)
FIG = OUT_DIR / "figures"
random.seed(SEED); np.random.seed(SEED)

## Step 2: The lab measurements
One row per measured piece (thread length, caliper diameter). Per clip: mean, SD, number of pieces and **standard error** (SD / √n). The median standard error over the clips is the **noise floor**: no method can be judged more finely than that.

In [ ]:
lab_pieces = labmod.load_lab_pieces(LABELS_FILE, LABELS_SHEET, LABEL_COLUMNS)
labels = labmod.clip_labels(lab_pieces)
floor = labmod.noise_floor(labels)
print(f"{len(lab_pieces)} measured pieces in {labels['clip'].nunique()} clips")
print(f"Noise floor: diameter {floor['diameter']:.2f} mm, length {floor['length']:.2f} mm")
display(labels.round(2))

## Step 3: The videos and what is already done
A video belongs to the clip with the same name (`901_1.MP4` ↔ `901_1`). The table shows what is already saved for each video.

In [ ]:
videos = [v for v in vid.list_videos(VIDEO_DIR) if not any(x in v.name for x in EXCLUDE)]
if ONLY_VIDEOS:
    videos = [v for v in videos if v.stem in ONLY_VIDEOS]
_info = {}
def info_of(v):
    v = str(v)
    if v not in _info:
        _info[v] = vid.probe(v)
    return _info[v]

def clip_of(video_stem):
    return video_stem      # the file name is the clip name

def has_clicks(v):
    return pipeline.load_clicks(RESULTS_DIR / v.stem / "clicks.json").get("clicked", {}).get("hole_A") is not None

def status():
    rows = []
    for v in videos:
        d = RESULTS_DIR / v.stem
        c = pipeline.load_clicks(d / "clicks.json")
        rows.append(dict(video=v.name, lab_pieces=int((lab_pieces["clip"] == clip_of(v.stem)).sum()),
                         clean_die=(d / "clean_die.png").exists(), clicked=has_clicks(v), refined="refined" in c,
                         tracked=(d / "tracking.json").exists(), measured=(d / "pieces.csv").exists()))
    return pd.DataFrame(rows)

print(f"{len(videos)} videos in {VIDEO_DIR}")
display(status())

## Step 4: Clean-die pictures (only the missing ones)
For each pixel, the median over about 70 frames: moving product disappears and the die plate stays, so the holes and bolts are easy to click. About 15 s per video.

In [ ]:
for v in videos:
    p = RESULTS_DIR / v.stem / "clean_die.png"
    if not p.exists():
        p.parent.mkdir(parents=True, exist_ok=True)
        t0 = time.time()
        Image.fromarray(vid.clean_die_image(v)).save(p)
        print(f"{v.name}: clean die picture made in {time.time() - t0:.0f} s")
print("All clean-die pictures are ready.")

## Step 5: Reference points (the click tool opens only if needed)
- **Every video already has saved points:** nothing to do; the cell says so and you continue.
- **Some videos have none:** the click tool appears, opened at the first such video. For each one click
  1. **hole A** (lower yellow spot, where our pieces come out),
  2. **hole B** (upper yellow spot),
  3. the **bolt(s)** you can see (usually the dark circle to the right),

  then press **Save & next**. **Copy points from previous video** saves time when the camera did not move.
- Zoom with the toolbar **magnifier**, and click it again before clicking points (clicks are ignored while zoom is on).
- To look at or change saved points, set `SHOW_CLICKER = True` in Step 1.

The points are saved in `results/<video>/clicks.json` and never asked for again.

In [ ]:
to_click = [v for v in videos if not has_clicks(v)]
clicker = None
if to_click or SHOW_CLICKER:
    get_ipython().run_line_magic("matplotlib", "widget")
    clicker = MultiVideoClicker(videos, RESULTS_DIR, info_of)
    clicker.show()
    print(f"Videos still without points: {[v.stem for v in to_click] or 'none'}")
else:
    print(f"All {len(videos)} videos already have saved points: nothing to click. "
          "(Set SHOW_CLICKER = True in Step 1 to look at or change them.)")

In [ ]:
%matplotlib inline
if clicker is not None:
    plt.close(clicker.fig)
ready = [v for v in videos if has_clicks(v)]
missing = [v.stem for v in videos if not has_clicks(v)]
if missing:
    print("Still without points (skipped below; go back to Step 5 to click them):", missing)
print(f"{len(ready)} of {len(videos)} videos are ready.")

## Step 6: Exact hole centres and the millimetre scale
Holes A and B are moved to the median centre of their fresh rope ends (done once, saved as `refined` in `clicks.json`). The scale comes from holes A and B (55 mm apart) or from both bolts (65 mm apart). The `check_…_mm` columns compare the other points with the die-plate map: a few tenths of a mm is fine, several mm means a misplaced click.

In [ ]:
mappings = {}
for v in ready:
    p = RESULTS_DIR / v.stem / "clicks.json"
    c = json.loads(p.read_text())
    if "refined" not in c:
        refined, _ = pipeline.refine_points(v, info_of(v), c["clicked"], log=lambda s: print(f"  {v.stem}: {s}"))
        c["refined"] = {k: list(map(float, xy)) for k, xy in refined.items()}
        p.write_text(json.dumps(c, indent=1))
    mappings[v.stem] = experiment.video_mapping(c)

rows = []
for v in ready:
    m, name = mappings[v.stem]
    rows.append(dict(video=v.stem, reference=name, px_per_mm=m["px_per_mm"] if m else np.nan,
                     **{f"check_{k}_mm": round(d, 2) for k, d in (m["offset_mm"].items() if m else [])
                        if k not in m["points_used"]}))
ref_table = pd.DataFrame(rows)
display(ref_table.round(3))

## Step 7–8: Find and track every piece (SAM 2.1)
For each video: 360 × 360 px crop around hole A → every knife cut from the yellowness "blink" → automatic box on each fresh rope end → SAM 2 follows the piece: **RED** = cut moment, **BLUE** = first detached frame.

- A video already tracked with the same settings is **re-used** ("Re-using saved tracking").
- **SAM 2 is loaded only when a video still needs tracking** (it takes about 20 s and 2–3 GB of GPU memory).
- If the run stops, run the cell again: finished videos are kept; only the unfinished one starts over.
- Changing `MAX_CUTS`, `N_TRACK` or `USE_BRIGHTNESS` makes the affected videos track again automatically.

In [ ]:
_predictor = None
def sam2_predictor():
    # loads SAM 2.1 the first time a video needs tracking
    global _predictor
    if _predictor is None:
        from sam2.build_sam import build_sam2_video_predictor
        _predictor = build_sam2_video_predictor(SAM2_CONFIG, str(SAM2_CHECKPOINT), device=DEVICE)
        track.quiet_sam2()
        print(f"    SAM 2.1 large loaded on {DEVICE}")
    return _predictor

track_rows = []
for i, v in enumerate(ready):
    out = RESULTS_DIR / v.stem
    c = json.loads((out / "clicks.json").read_text())
    bright = float(c.get("brightness", 1.0)) if USE_BRIGHTNESS else 1.0
    print(f"[{i + 1}/{len(ready)}] {v.name}")
    t0 = time.time()
    table, prep = pipeline.track_video_cached(sam2_predictor, v, out, c["refined"]["hole_A"], CROP_SIZE, N_TRACK,
                                              MAX_CUTS, DEVICE, force=FORCE_TRACKING,
                                              log=lambda s: print("   ", s), brightness=bright)
    track_rows.append(dict(video=v.stem, cuts_found=len(table), kept_by_tracking=int(table["keep"].sum()),
                           cut_rate_video=round(prep["cuts"]["cut_hz"], 2),
                           window=json.loads((out / "tracking.json").read_text())["n_track"],
                           minutes=round((time.time() - t0) / 60, 1)))
track_table = pd.DataFrame(track_rows)
display(track_table)

## Step 9: Measure every RED and BLUE mask
**Length** along the bend (like the lab's thread) and **diameter** = median diameter of the biggest circles that fit inside, each measured at 4 rotations. The three bad-mask checks (ragged outline, one tube?, unstable) are computed at the same time. About 1–2 minutes per video on several CPU processes; saved in `results/<video>/pieces.csv`.

In [ ]:
per_video = {}
for i, v in enumerate(ready):
    m, name = mappings[v.stem]
    px_to_mm = m["px_to_mm"] if m else experiment.nominal_mapping()
    ppm = m["px_per_mm"] if m else experiment.NOMINAL_PX_PER_MM
    t0 = time.time()
    per_video[v.stem] = experiment.measure_video(RESULTS_DIR / v.stem, px_to_mm, ppm, name, n_jobs=N_JOBS,
                                                 force=FORCE_MEASURE)
    print(f"[{i + 1}/{len(ready)}] {v.stem}: {len(per_video[v.stem])} masks ({time.time() - t0:.0f} s)")

## Step 10: The length ratio of every mask
For every clip and mask type (RED / BLUE):

`length_ratio = length of this mask ÷ length of the longest good mask of the clip`

- "Good" = passes the three bad-mask checks. Only a good mask can be "the longest", so a merged or leaked mask (which can be *longer* than the real piece) cannot set the bar.
- `length_ratio_p95` uses the 95th-percentile length instead of the longest: a safer bar if one odd mask is still too long.

> **Example.** The longest good BLUE mask of a clip is 10 mm. A mask of 7 mm has ratio 0.7, so it is kept by the 0.5, 0.6 and 0.7 selections, and dropped by 0.8 and above.

In [ ]:
pieces = experiment.add_length_ratio(experiment.collect_pieces(per_video, clip_of))
print(f"{len(pieces)} masks in {pieces['video'].nunique()} videos "
      f"({(pieces['key'] == 'red').sum()} RED, {(pieces['key'] == 'blue').sum()} BLUE); "
      f"{pieces['used'].sum()} pass the bad-mask checks")
display(pieces[pieces["used"]].groupby("key")[["length_mm", "length_ratio", "length_ratio_p95"]]
        .describe(percentiles=[0.1, 0.5, 0.9]).round(2).T)

## Step 11: The idea on one clip
**Top:** lengths of the good BLUE masks of one clip, with the longest and three thresholds. **Bottom:** masks from the shortest-looking to the longest. The short ones are mostly pieces pointing toward the camera; their true length is hidden. Run the cell again with another `EXAMPLE_CLIP` to see other clips.

In [ ]:
ex_clip = EXAMPLE_CLIP or sorted(pieces["clip"].unique())[0]
for key in experiment.KEYS:
    dl0.plot_length_idea(pieces, ex_clip, key, path=FIG / f"length_idea_{ex_clip}_{key}.png"); plt.show()

## Step 12: The piece selections and the clip averages
| Selection | Pieces averaged per clip |
|---|---|
| every mask | every measured mask, also the bad ones |
| good masks | masks that pass the three bad-mask checks (Experiment 1's main variant) |
| ≥0.5 longest … ≥0.9 longest | good masks at least 50 … 90 % as long as the longest good mask |
| longest only | only the longest good mask (one piece per clip, so no SD) |
| ≥0.5 p95 … ≥1.0 p95 | the same, against the 95th-percentile length |

Fewer pieces per clip = a noisier clip average, so a stricter selection is not automatically better. The figure shows how many pieces each selection keeps.

In [ ]:
selections = experiment.length_selections(LENGTH_THRESHOLDS, LENGTH_REFERENCES)
cm = dl0.clip_means(pieces, selections, labels)
dl0.plot_pieces_kept(cm, selections, path=FIG / "pieces_kept.png"); plt.show()
display(cm.pivot_table(index="clip", columns=["key", "selection"], values="n_pieces")
        .reindex(columns=list(selections), level=1).astype("Int64"))

## Step 13: DL-0, one learned number
**The model.** For each mask type and target (diameter, length):

`k = average(lab clip means) ÷ average(video clip means)`, over the **training clips**, then `prediction = k × video clip mean` on the **test clips**.

> **Example.** On the training clips the video's diameter averages 10.8 mm and the lab's 9.0 mm, so k = 9.0 ÷ 10.8 = **0.833**. A test clip where the video says 11.2 mm is predicted as 0.833 × 11.2 = **9.33 mm**.

**Why:** if the camera's error is mostly one constant scale (perspective, the mask edge, pieces seen at an angle), one number fixes it. Any deeper model has to beat this.

**Test: leave one setting out.** A clip is never used to learn the *k* that predicts it. Learn *k* on the 15 clips of 3 settings, predict the 5 clips of the 4th; repeat for each setting, so every clip is predicted once. This answers "does it work at a setting it has never seen?", like a new production run nobody measured by hand.

**Compared with: no training (k = 1),** the geometric value itself, as in Experiment 1.

**Choosing the selection fairly ("chosen on training clips").** Picking the best selection by looking at the test errors would flatter the result. So for every test fold, each selection is tried on the **training clips only** (leaving one of their settings out at a time), the best one is kept, and only then is the test clip predicted. This is the honest number to report; the per-selection rows are the ablation.

In [ ]:
pred, chosen = dl0.predict_all(cm, SPLITS)
print(f"{len(pred)} predictions")
k_table = (pred[(pred["method"] == dl0.DL0) & (pred["split"] == "leave one setting out") & (pred["selection"] == "good masks")]
           .drop_duplicates(["key", "target", "fold"]).pivot_table(index="fold", columns=["key", "target"], values="k"))
print("k learned in each leave-one-setting-out fold (good masks); the fold name is the setting left out:")
display(k_table.round(3))

## Step 14: Errors for every selection
**MAE** (mean absolute error per clip, mm) for every selection, mask type and method. Lower is better; compare with the noise floor of Step 2. The full table (RMSE, MAPE, bias, limits of agreement, R², correlation, SD error, 95 % intervals) is saved in `metrics.csv`.

| Measure | Meaning |
|---|---|
| **MAE** | average size of the error per clip (mm) |
| **bias** | average signed error: is the prediction always too big or too small? |
| R² | 1 = perfect; 0 = no better than always guessing the lab average; can be negative |

In [ ]:
met = dl0.metrics(pred)
met["method_split"] = np.where(met["method"] == dl0.GEOMETRIC, "geometric", "DL-0, " + met["split"])
order = list(selections) + [dl0.CHOSEN]
for t in dl0.TARGETS:
    print(f"\n{t.upper()}: MAE per clip (mm); noise floor {floor[t]:.2f} mm")
    display(met[met["target"] == t].pivot_table(index="selection", columns=["key", "method_split"], values="MAE")
            .reindex(order).round(2))

## Step 15: The ablation, as a picture
x axis: from every mask (left) to the longest only (right). Solid lines: thresholds relative to the longest mask; dashed: relative to the 95th-percentile length. Dotted horizontal lines: the selection chosen on training clips.

How to read it: if the grey line (no training) drops to the right, choosing long pieces fixes the foreshortening of length. If DL-0 is flat, the learned *k* already absorbs most of it.

*Note:* if all selections are about equally good, the "chosen on training clips" choice is driven by noise and can come out slightly worse than a single fixed selection. That itself says the selection does not matter much.

In [ ]:
for key in experiment.KEYS:
    dl0.plot_ablation(met, key, selections, path=FIG / f"ablation_{key}.png"); plt.show()

## Step 16: Is DL-0 really better than the geometric value?
For the same clips: the MAE difference (DL-0 − geometric), its 95 % paired bootstrap interval (2000 resamples of the clips), and the Wilcoxon signed-rank p-value. **DL-0 counts as better only if its MAE is lower and the whole interval is below 0.**

In [ ]:
vs = dl0.versus_geometric(pred)
for t in dl0.TARGETS:
    print(f"\n{t.upper()}")
    display(vs[vs["target"] == t].drop(columns="target").round(3))

## Step 17: The honest result: selection chosen on training clips
Which selection each fold chose (with the training clips only), and the full error table of these rows. **"DL-0, leave one setting out" is the main number of this notebook.**

In [ ]:
display(chosen.groupby(["method", "split", "key", "target"])["chosen"]
        .agg(lambda s: ", ".join(f"{k} ×{n}" for k, n in s.value_counts().items())).to_frame("chosen in the folds"))
cols = ["method", "split", "key", "target", "clips", "MAE", "MAE_CI_low", "MAE_CI_high", "RMSE", "MAPE", "bias",
        "limits_of_agreement", "R2", "correlation", "noise_floor", "SD_MAE"]
display(met[met["selection"] == dl0.CHOSEN][cols].sort_values(["target", "key", "method", "split"]).round(3))

## Step 18: Agreement plots (DL-0, leave one setting out, selection chosen on training clips)
- **Left:** prediction vs lab, one point per clip, coloured by setting (bars = lab standard error). On the dashed line they agree exactly.
- **Right:** Bland–Altman: difference (prediction − lab) against their average, with the bias (solid) and 95 % limits of agreement (dashed).

In [ ]:
main = pred[(pred["method"] == dl0.DL0) & (pred["split"] == "leave one setting out") & (pred["selection"] == dl0.CHOSEN)]
main = main.assign(variant="main")
for key in experiment.KEYS:
    if len(main[main["key"] == key]):
        experiment.plot_agreement(main, key, variant="main", path=FIG / f"agreement_{key}.png",
                                  title=f"DL-0 — {key.upper()} masks — leave one setting out, selection chosen on training clips")
        plt.show()

## Step 19: Save everything
| File (`results/dl0/`) | Content |
|---|---|
| `pieces.csv` | one row per mask: measurements, bad-mask check, **`length_ratio`**, `length_ratio_p95` (DL-1 can filter on these) |
| `clip_means.csv` | per selection, clip and mask type: pieces used, video mean / SD, lab values |
| `predictions.csv` | every clip prediction: method, split, selection, fold, *k*, prediction, lab, error |
| `metrics.csv` | MAE (+ 95 % interval), RMSE, MAPE, bias, limits of agreement, R², correlation, noise floor, SD error |
| `dl0_vs_geometric.csv` | MAE difference, 95 % interval, Wilcoxon p, "DL-0 better" |
| `chosen_selections.csv` | the selection chosen in every fold |
| `settings.json`, `figures/` | all settings, every figure |

In [ ]:
keep_cols = ["video", "clip", "setting", "cut", "key", "frame", "frames_to_cut", "used", "mask_problem",
             "length_mm", "length_ratio", "length_ratio_p95", "width_mm", "width_max_mm", "width_mean_mm",
             "feret_mm", "area_mm2", "straightness", "roughness_pct", "tube_fill", "shape_stability_pct",
             "px_per_mm", "reference", "mask_file"]
pieces[[c for c in keep_cols if c in pieces]].to_csv(OUT_DIR / "pieces.csv", index=False)
cm.to_csv(OUT_DIR / "clip_means.csv", index=False)
pred.to_csv(OUT_DIR / "predictions.csv", index=False)
met.drop(columns="method_split").to_csv(OUT_DIR / "metrics.csv", index=False)
vs.to_csv(OUT_DIR / "dl0_vs_geometric.csv", index=False)
chosen.to_csv(OUT_DIR / "chosen_selections.csv", index=False)
settings = dict(videos=[v.name for v in ready], labels_file=str(LABELS_FILE), labels_sheet=LABELS_SHEET,
                crop_size=CROP_SIZE, n_track=N_TRACK, max_cuts=MAX_CUTS, use_brightness=USE_BRIGHTNESS,
                max_roughness_pct=measure.MAX_ROUGHNESS_PCT, tube_fill_range=list(measure.TUBE_FILL_RANGE),
                max_stability_pct=measure.MAX_STABILITY_PCT, length_thresholds=list(LENGTH_THRESHOLDS),
                length_references=list(LENGTH_REFERENCES), splits=list(SPLITS), selections=selections,
                tracking_version=pipeline.TRACKING_VERSION, measure_version=experiment.MEASURE_VERSION,
                sam2_checkpoint=SAM2_CHECKPOINT.name)
(OUT_DIR / "settings.json").write_text(json.dumps(settings, indent=1))
for f in sorted(OUT_DIR.glob("*")):
    print(f.relative_to(RESULTS_DIR))